In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def visualize_vog_data(file_path):
    """
    단일 VOG 안구 운동 데이터를 로드하고 3가지 핵심 뷰를 시각화합니다.
    (Null Byte 메모리 절단 방지 및 다중 인코딩 자가 치유 파이프라인 적용)
    """
    path_obj = Path(file_path)
    if not path_obj.exists():
        print(f"[Error] 파일을 찾을 수 없습니다: {path_obj.resolve()}")
        return False

    # 0. 아키텍처 개선: 파일 경로(Path) 기반 동적 메타데이터 추출 (Traceability 확보)
    # 예: HC_csv_24_25 -> HC (건강 대조군) / MCI_csv_25_26 -> MCI (경도인지장애)
    group_name = path_obj.parent.parent.name if len(path_obj.parents) >= 2 else "Unknown_Group"

    # 예: 20240329_084111_1324721_ (환자 세션 ID)
    session_id = path_obj.parent.name

    # 예: 'PD VOG -_Horizontal Saccade B (anti)' -> 'Horizontal Saccade B (anti)'
    task_name = path_obj.stem.replace("PD VOG -_", "").replace("PD VOG -", "").strip()

    try:
        header_idx = -1
        header_columns = []
        raw_lines = []

        encodings_to_try = ['utf-16', 'utf-16le', 'utf-8-sig', 'cp949']

        for enc in encodings_to_try:
            try:
                with open(path_obj, 'r', encoding=enc, errors='replace') as f:
                    lines = f.readlines()

                for i, line in enumerate(lines):
                    line_clean = line.replace('\x00', '').lower()
                    if 'lh' in line_clean and 'rh' in line_clean and 'target' in line_clean:
                        header_idx = i
                        raw_lines = lines
                        header_columns = [col.replace('\x00', '').strip() for col in line.split(',')]
                        break

                if header_idx != -1:
                    break
            except UnicodeError:
                continue

        if header_idx == -1:
            print("[Skip] VOG 데이터 헤더(LH, RH, Target)를 찾을 수 없는 파일입니다.")
            return False

        parsed_data = []
        for line in raw_lines[header_idx + 1:]:
            line_clean = line.replace('\x00', '').strip()
            if not line_clean:
                continue

            row_values = [val.strip() for val in line_clean.split(',')]

            if len(row_values) < len(header_columns):
                row_values.extend([''] * (len(header_columns) - len(row_values)))
            elif len(row_values) > len(header_columns):
                row_values = row_values[:len(header_columns)]

            parsed_data.append(row_values)

        df = pd.DataFrame(parsed_data, columns=header_columns)
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(how='all').reset_index(drop=True)

    except Exception as e:
        print(f"[Error] 파싱 중 치명적 오류: {e}")
        return False

    time_col = next((col for col in df.columns if 'time' in str(col).lower() or str(col).lower() == 't'), None)
    if not time_col:
        time_col = df.columns[0]
    time_sec = df[time_col]

    target_v_col = next((c for c in df.columns if 'targetv' in str(c).lower() or 'target_v' in str(c).lower()), None)
    target_h_col = next((c for c in df.columns if 'targeth' in str(c).lower() or 'target_h' in str(c).lower()), None)

    target_col = None
    if target_v_col and df[target_v_col].abs().sum() > 0:
        target_col = target_v_col
    elif target_h_col:
        target_col = target_h_col
    elif target_v_col:
        target_col = target_v_col

    if not target_col:
        print(f"[Error] 타겟(Target) 컬럼 탐색 실패.")
        return False

    direction_str = "Vertical" if 'v' in str(target_col).lower() else "Horizontal"

    search_l = 'lv' if direction_str == "Vertical" else 'lh'
    search_r = 'rv' if direction_str == "Vertical" else 'rh'

    eye_col_l = next((c for c in df.columns if str(c).lower() == search_l), None)
    eye_col_r = next((c for c in df.columns if str(c).lower() == search_r), None)

    if not eye_col_l or not eye_col_r:
        print(f"[Error] 필수 측정 컬럼({search_l}, {search_r}) 누락.")
        return False

    df['Error_L'] = df[eye_col_l] - df[target_col]
    df['Error_R'] = df[eye_col_r] - df[target_col]

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

    # 수정된 부분: 추출한 메타데이터(환자군, 세션, 검사명)를 그래프 최상단에 직관적으로 각인
    title_str = f"[{group_name}] Session: {session_id} \n Task: {task_name} ({direction_str} Analysis)"
    fig.suptitle(title_str, fontsize=18, fontweight='bold')

    axes[0].plot(time_sec, df[target_col], label=f'Target ({target_col})', color='red', linestyle='--', linewidth=2)
    axes[0].plot(time_sec, df[eye_col_l], label=f'Left Eye ({eye_col_l})', color='blue', alpha=0.7)
    axes[0].plot(time_sec, df[eye_col_r], label=f'Right Eye ({eye_col_r})', color='green', alpha=0.7)
    axes[0].set_title(f'1. Raw Waveform: Target vs Eye Movement', fontsize=14)
    axes[0].set_ylabel('Position')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, linestyle=':', alpha=0.6)

    axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[1].plot(time_sec, df['Error_L'], label=f'Error ({eye_col_l} - Target)', color='purple', alpha=0.8)
    axes[1].plot(time_sec, df['Error_R'], label=f'Error ({eye_col_r} - Target)', color='orange', alpha=0.8)
    axes[1].set_title('2. Derived Feature: Eye Tracking Error', fontsize=14)
    axes[1].set_ylabel('Error Distance')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, linestyle=':', alpha=0.6)

    noise_col_l = 'LH' if direction_str == "Vertical" else 'LV'
    noise_col_r = 'RH' if direction_str == "Vertical" else 'RV'

    actual_noise_col_l = next((c for c in df.columns if str(c).lower() == noise_col_l.lower()), None)
    actual_noise_col_r = next((c for c in df.columns if str(c).lower() == noise_col_r.lower()), None)

    if actual_noise_col_l and actual_noise_col_r:
        axes[2].plot(time_sec, df[actual_noise_col_l], label=f'Left Eye ({actual_noise_col_l})', color='gray', alpha=0.7)
        axes[2].plot(time_sec, df[actual_noise_col_r], label=f'Right Eye ({actual_noise_col_r})', color='brown', alpha=0.7)
        axes[2].set_title(f'3. Outlier/Noise Monitoring (Movement in non-target axis)', fontsize=14)
        axes[2].set_xlabel('Time (sec)', fontsize=12)
        axes[2].set_ylabel('Position')
        axes[2].legend(loc='upper right')
        axes[2].grid(True, linestyle=':', alpha=0.6)
    else:
        axes[2].set_title(f'3. Outlier/Noise Monitoring (Data Not Available for {noise_col_l}, {noise_col_r})', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Jupyter 환경에서 그림을 즉시 렌더링
    plt.show()

    # 메모리 누수(OOM: Out Of Memory) 방지용 자원 해제 (매우 중요)
    plt.close(fig)

    return True

def visualize_directory_recursively(base_dir_path):
    """
    주어진 디렉토리와 모든 하위(Sub) 디렉토리를 순회하며
    발견된 모든 CSV 파일의 VOG 데이터를 연속으로 시각화합니다.
    """
    base_path = Path(base_dir_path)

    if not base_path.exists() or not base_path.is_dir():
        print(f"[Error] 유효한 디렉토리가 아닙니다: {base_path.resolve()}")
        return

    # rglob('*.csv')를 사용하여 하위 디렉토리까지 재귀적으로(Recursive) 탐색
    csv_files = list(base_path.rglob('*.csv'))

    if not csv_files:
        print(f"[Info] '{base_path.name}' 디렉토리 내에 CSV 파일이 존재하지 않습니다.")
        return

    print("=" * 70)
    print(f"🚀 VOG 데이터 배치(Batch) 처리 시작")
    print(f"- 대상 폴더: {base_path.resolve()}")
    print(f"- 발견된 CSV 파일: 총 {len(csv_files)}개")
    print("=" * 70)

    success_count = 0
    for i, file_path in enumerate(csv_files, 1):
        print(f"\n▶ [{i}/{len(csv_files)}] Processing: {file_path.name}")
        print(f"  └ Path: {file_path.parent}")

        # 시각화 실행 및 성공 여부 확인
        is_success = visualize_vog_data(file_path)
        if is_success:
            success_count += 1

    print("\n" + "=" * 70)
    print(f"✅ 모든 처리 완료! (성공: {success_count} / 전체: {len(csv_files)})")
    print("=" * 70)

# 실행 엔트리 포인트
if __name__ == "__main__":
    # 프로젝트 루트 내의 최상위 데이터 폴더를 지정하세요. (예: "data" 또는 ".")
    # '.'을 입력하면 현재 노트북 파일이 있는 폴더의 모든 하위 폴더를 뒤집니다.
    target_directory = "data/sample_csv"

    # 디렉토리 순회 시각화 파이프라인 가동
    visualize_directory_recursively(target_directory)